# Check dilution calculations

Exploratory source notebook; known incomplete cells remain. See the method README before running.

Source: `rjost1/FluvialSeismology_Analysis`, `Dilution/dilutionUtils_testing.ipynb`, commit `fce22815c02abd61bebdff692fd49c128feb5f3a`. Calculations are preserved; imports and paths have been adapted. Stored source outputs have been cleared.

In [ ]:
from pathlib import Path
import os
import sys

PROJECT_ROOT = next(
    (p for p in [Path.cwd(), *Path.cwd().parents] if (p / "project_paths.py").is_file()),
    None,
)
if PROJECT_ROOT is None:
    raise RuntimeError("Start Jupyter inside the mora-fluvial-analysis repository.")
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
from project_paths import method_path, result_path
METHOD_ROOT = PROJECT_ROOT / 'salt_dilution'
os.chdir(METHOD_ROOT)


In [ ]:
from datetime import time

from salt_dilution.scripts.processing import dilution_calculations as dil

In [ ]:
files, injection, calibration = dil.load_config("config/202504_trial1_solution_config.ini")

In [ ]:
csv_files ="."
search_string = "data/examples/20250709_*_790478.csv"
calib_file = 'data/examples/calibration.csv'
cal_data = dil.read_csv(calib_file)


In [ ]:
x = dil.extract_fields(cal_data)

In [ ]:
# This is important
# Mixing the Primary sample
# note that on 07/09 the injection sample used to calibrate was taken from the second injection but used to calibrate both trials.
Vo = 12000 # mL of H20 collected stream
rho_salt = 2.165 # density of NaCl in g/cm3
mass_salt = 4000 # g
Vc = x['Vol_secondary_mL'][0]
vol_calibration_sample = 100 # ml
Vnew = Vo +  mass_salt / rho_salt
vol_solution_injected = Vnew - vol_calibration_sample # ml
injection_conc = (mass_salt) / (Vo) # g/ml

print(injection_conc, Vnew)

In [ ]:
RC_values = dil.calibration_RC(injection_conc, x['Vol_cal_solution_mL'], Vc)

In [ ]:
# returns np.float64(5.728194064066105e-07) with the given calibration file
# is that right?
slope_k = dil.calculate_k(RC_values, x['actual_CD_uS_per_cm'])

In [ ]:
data = dil.get_file_paths(csv_files, search_string)
data

In [ ]:
ec_data_trial_1 = dil.extract_fields(dil.read_csv(data[0]))
ec_data_trial_2 = dil.extract_fields(dil.read_csv(data[1]))

In [ ]:
# better that get_measurement_values function
def time_diff_sec(start:str, end:str):
    """ Calculate the diffence between to times in seconds.

        Parameters:
            start (string): "H:M:S"
            end (string): "H:M:S"
        Returns:
            float: difference in seconds between the inputs.
        """
    t1 = datetime.strptime(start,"%H:%M:%S")
    t2 = datetime.strptime(end,"%H:%M:%S")
    diff_seconds = (t2-t1).total_seconds()

    return int(diff_seconds)

In [ ]:
msnt1_start = time(14,50,33) #"14:50:33"
msnt1_end = time(14,58,6) #"14:58:06"
# Pylance prefers it like this
result1 = dil.get_measurement_time_values(msnt1_start, msnt1_end, ec_data_trial_1, show_data=True)
if result1 is not None:
    background1, trial1, trial_time1 = result1
else:
    print("Start or end time not found in ec_data_trial_1['Date Time']")

In [ ]:
time_list1 = trial_time1
time_diffs1, time_elapsed1 = dil.compute_time_differences_in_seconds(time_list1)
EC_bg1 = dil.get_background_ec(ec_data=background1,start_time=None,end_time=None)
Q_1 = dil.calculate_Q(EC_bg1,trial1,slope_k,time_diffs1,vol_solution_injected)

In [ ]:
Q_1

In [ ]:

msnt2_start = time(15,17,38) #"15:17:38"
msnt2_end = time(15,24,41) #"15:24:41"
# Pylance prefers it like this
result2 = dil.get_measurement_time_values(msnt2_start, msnt2_end, ec_data_trial_2, show_data=True)
if result2 is not None:
    background_2, trial_2, trial_time_2 = result2
else:
    print("Start or end time not found in ec_data_trial_2['Date Time']")

In [ ]:
time_list2 = trial_time_2
time_diffs2, time_elapsed2 = dil.compute_time_differences_in_seconds(time_list2)
EC_bg2 = dil.get_background_ec(ec_data=background_2,start_time=None,end_time=None)
Q_2 = dil.calculate_Q(EC_bg2,trial_2,slope_k,time_diffs2,vol_solution_injected)

In [ ]:
Q_2